In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("spotify_millsongdata.csv")

In [3]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [4]:
df.tail(5)

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [5]:
df.shape

(57650, 4)

In [6]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [7]:
df =df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [8]:
df.head(10)

,artist,song,text
0,Irving Berlin,What Chance Have I With Love?,"Love is beautiful, love is swell \r\nLove is ..."
1,Kiss,Almost Human,"Ahh, ha \r\nI'm almost human, can't help feel..."
2,America,Submarine Ladies,Why do you laugh at me when you know I really ...
3,Peter Gabriel,Heroes,"I, I will be king \r\nAnd you, you will be qu..."
4,David Bowie,Drive-in Saturday,(uh uh aaah) Let me put my arms around your he...
5,George Harrison,Far East Man,While the world wages war \r\nIt gets harder ...
6,Lou Reed,Senselessly Cruel,When I was p poor young boy at school \r\nGir...
7,Vertical Horizon,Children's Lullaby,Little girl was down by the waterside \r\nThe...
8,Rammstein,Kuess Mich,Sie haelt immer still \r\nWeil sie gefingert ...
9,Selena Gomez,Revival,[Intro] \r\nI dive into the future \r\nBut I...


In [9]:
df['text'][0]

"Love is beautiful, love is swell  \r\nLove is as sweet as a nut  \r\nLove is grander than tongue can tell  \r\nLove is remarkable, but  \r\nLook at what it did to Antony  \r\nIt made a fool out of Antony  \r\nIf love could do that to Antony  \r\nWhat chance have I with love?  \r\nLook at what it did to Romeo  \r\nIt dealt poor Romey an awful blow  \r\nIf love could do that to Romeo  \r\nWhat chance have I with love?  \r\nLook at what it did to Samson  \r\n'Til he lost his hair he was brave  \r\nIf a haircut could weaken Samson  \r\nThey could murder me with a shave  \r\nLook at what it did to Bonaparte  \r\nHe lost his head when he lost his heart  \r\nIf he kicked over the apple cart  \r\nWhat chance have I  \r\nAn ordinary guy  \r\nWhat chance have I with love\r\n\r\n"

In [11]:
# df = df.sample(5000)

In [12]:
df.shape

(5000, 3)

Text Cleaning/ Text Preprocessing

In [20]:
df['text'] = df['text'].str.lower().replace(r'^\w\s', ' ').replace(r'\n', ' ', regex = True)

In [23]:
import nltk
from nltk.stem.porter import PorterStemmer

# Ensure necessary NLTK resources are downloaded
nltk.download('punkt')

# Initialize the PorterStemmer
stemmer = PorterStemmer()

# Define the tokenization function with stemming
def tokenization(txt):
    # Tokenize the text into words
    tokens = nltk.word_tokenize(txt)
    # Apply stemming to each token
    stemming = [stemmer.stem(w) for w in tokens]
    # Return the stemmed tokens as a single string
    return " ".join(stemming)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dhriti\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [25]:
df['text'] = df['text'].fillna('')

In [26]:
df['text'] = df['text'].apply(lambda x: tokenization(x))

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)

In [29]:
similarity[0]

array([1.        , 0.01064513, 0.25344423, ..., 0.02799737, 0.00205191,
       0.11139033])

In [30]:
df[df['song'] == 'Crying Over You']

,artist,song,text
1881,UB40,Crying Over You,cri over you in the morn cri over you in the e...


In [31]:
def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similarity[idx])),reverse=True,key=lambda x:x[1])
    
    songs = []
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)
        
    return songs

In [32]:
recommendation('Crying Over You')

["Don't Cry",
 "Don't Cry Now",
 'No Woman',
 'The Same Love That Made Me Laugh',
 "Big Boys Don't Cry",
 "You Won't See Me Cry",
 'Cry Cry Cry',
 "Why Can't I Cry",
 'Bird Song',
 'When The Time Comes',
 "Don't Cry No More",
 "Call Me When You're Sober",
 "I'm So Lonesome I Could Cry",
 'I Cried Again',
 'Do It Again',
 "Don't Cry",
 "Don't Mind If I Do",
 'I Heard You Crying In Your Sleep',
 'Cry Of The Broken',
 "I'm So Lonesome I Could Cry"]

In [33]:
import pickle
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))